# Laboratory 5A — Understand the multi-volume QHA dataset

The expensive part of Laboratory 5 has already been calculated. Your first job
is **not** to run anything: it is to understand exactly what information is in
the supplied CRYSTAL calculations and why a quasi-harmonic model needs it.


In [ ]:
from pathlib import Path
import pandas as pd
from mineral_science_tools.qha_lab import reference_series_table

series = reference_series_table("reference-results/raw")
series


## 1. What was actually calculated?

Inspect the table and then open one `.d12` and one `.out` file as text.

### Question 1

What is the `SCELPHONO` expansion matrix? How many primitive-cell repetitions
does it contain, how many atoms are in the phonon supercell, and how many
q-points are calculated?

**Your answer:**  
_Write your answer here._


### Question 2

Which exchange–correlation functional and which basis set were used?

The calculation is deliberately not using the cheapest possible basis: the
reference data already use `POB-TZVP-REV2`, a triple-zeta valence + polarization
basis designed for periodic solids.

**Your answer:**  
_Write your answer here._


## 2. The volume interval

The seven structures span both compression and expansion around the central
reference state.


In [ ]:
series[[
    "V/Vref label",
    "a conventional (A)",
    "V conventional (A^3)",
    "V/Vref from cell",
]]


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
ax.scatter(series["V/Vref label"], series["V conventional (A^3)"])
ax.axvline(1.0, linewidth=0.8, linestyle="--")
ax.set_xlabel("V / Vref")
ax.set_ylabel("Conventional cell volume (Å³)")
fig.tight_layout()


### Question 3 — Why do we need expanded volumes?

Suppose we had calculated only the compressed states
$0.91V_\mathrm{ref}$, $0.94V_\mathrm{ref}$ and $0.97V_\mathrm{ref}$.

Would that be sufficient to model MgO at increasing temperature and ambient
pressure? Explain your answer in terms of **thermal expansion** and movement of
the minimum of $F(V,T)$.

**Your answer:**  
_Write 4–5 lines here._


## 3. Computational cost is part of model choice

The output reports 128 atoms and 2368 atomic orbitals for the $4\times4\times4$
phonon supercell. A more elaborate electronic-structure model has a real cost
when it must be repeated for every displaced geometry and every volume.

### Question 4

Why was a GGA functional such as PBE a reasonable teaching/production choice
for this large multi-volume phonon series?

Would replacing PBE with a hybrid functional automatically guarantee that
*every* predicted property is better? What would the computational trade-off
be?

**Your answer:**  
_Write 4–5 lines here._


### A note on basis sets

CRYSTAL distinguishes, among others, POB-DZVP-REV2 (double-zeta valence +
polarization) and POB-TZVP-REV2 (triple-zeta valence + polarization). A
triple-zeta valence basis gives the valence orbitals more radial flexibility,
but increases the number of basis functions and therefore the computational
cost.

The present reference calculations already use **POB-TZVP-REV2**. The remaining
systematic offset in equilibrium volume must therefore not be described simply
as "the basis was too small". Exchange–correlation approximation, basis-set
incompleteness, QHA limitations and numerical settings are distinct sources of
error.


## 4. Build the normalized Quantas QHA input

The list file is already ordered from the most compressed to the most expanded
calculation. The central $V/V_\mathrm{ref}=1.00$ calculation is the fourth file,
so its Python-style reference index is `3`.


In [ ]:
print(Path("reference-results/files.txt").read_text())


In [ ]:
!quantas qha inpgen reference-results/files.txt \
    --list \
    --interface crystal \
    --reference 3 \
    --output mgo_qha.yaml \
    --jobname "MgO PBE/POB-TZVP-REV2 4x4x4 QHA"


Quantas does more than concatenate files. For independent CRYSTAL outputs it
checks structural/q-mesh compatibility and attempts to establish phonon-mode
continuity across the volume series.

### Question 5

Read the generator message or inspect `mgo_qha.yaml`.

Why does mode continuity matter if we want to interpolate individual
$\nu_{\mathbf qj}(V)$ branches?

**Your answer:**  
_Write 3–4 lines here._


## 5. Inspect the sampled pressure support before running QHA


In [ ]:
!quantas qha inspect mgo_qha.yaml \
    --degree 3 \
    --eos BM3 \
    --report qha_inspect.log


### Question 6

Compare the polynomial and BM3 pressure estimates printed by `qha inspect`.
Does the sampled volume range comfortably support the pressure interval we plan
to use (0–10 GPa)?

Why is changing the fitting method **not** a valid substitute for missing
volume data?

**Your answer:**  
_Write 4–5 lines here._
